In [ ]:
from netgen.meshing import Mesh
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

import numpy as np

In [ ]:
def CapacitorGeometry(box_size, el_pos_wid, el_pos_h, el_neg_wid, 
                      el_neg_h, diel_wid, diel_h):

    air = MoveTo(0, 0).RectangleC(box_size, box_size).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    electrode_positive = MoveTo(0, 1).RectangleC(el_pos_wid, el_pos_h).Face()
    electrode_positive.edges.name = "electrode_positive"
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = MoveTo(0, -1).RectangleC(el_neg_wid, el_neg_h).Face()
    electrode_negative.edges.name = "electrode_negative"
    electrode_negative.faces.name = "electrode_negative"

    dielectric = MoveTo(0, 0).RectangleC(diel_wid, diel_h).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - electrode_positive - electrode_negative
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes_phi = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes_phi.TrialFunction()
    v = fes_phi.TestFunction()

    potential_gf = GridFunction(fes_phi)
    potential_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes_phi.FreeDofs())
    potential_gf.vec.data -= inv@a.mat * potential_gf.vec

    return potential_gf


def CapacitorErrorEstimator(mesh, phi_gf, epsr, FE_order):

    fes_E = HCurl(mesh, order=FE_order-1)
    E_gf = GridFunction(fes_E)
    E_gf.Set(-epsr*grad(phi_gf))

    E = -epsr*grad(phi_gf)

    error_gf = 1/epsr*(E - E_gf)*(E - E_gf)
    error_ZZ = Integrate(error_gf, mesh, VOL, element_wise=True)

    return error_gf, error_ZZ


In [ ]:
box_size = 30
el_pos_wid, el_pos_h = 5, 0.5
el_neg_wid, el_neg_h = 5, 0.5
diel_wid, diel_h = 4, 1.5

FE_order = 2

geo = CapacitorGeometry(box_size, el_pos_wid, el_pos_h, el_neg_wid, el_neg_h, diel_wid, diel_h)

In [ ]:
h_max = 3
mesh = CapacitorMesh(geo, h_max)

epsr_air, epsr_dielectric = 1.0, 2.0
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

FE_order = 2

potential_gf = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
error_gf, error_ZZ = CapacitorErrorEstimator(mesh, potential_gf, epsr, FE_order)

In [ ]:
Draw(error_gf, mesh, min=0, max=0.1);

In [ ]:
print(np.array(error_ZZ)[:10], "...")

maxerr = max(error_ZZ)
print ("maxerr = ", maxerr)

In [ ]:
for el in mesh.Elements():
    mesh.SetRefinementFlag(el, error_ZZ[el.nr] > 0.2*maxerr)

In [ ]:
mesh.Refine()
potential_gf = CapacitorSolver(mesh, FE_order, epsr)
error_gf, error_ZZ = CapacitorErrorEstimator(mesh, potential_gf, epsr, FE_order)

maxerr = max(error_ZZ)
print ("maxerr = ", maxerr)

In [ ]:
Draw(error_gf, mesh, min=0, max=0.1);